<style>
.note {padding: 12px 16px; border-left: 5px solid #2563eb; background: #eff6ff; margin: 10px 0;}
.warn {padding: 12px 16px; border-left: 5px solid #d97706; background: #fffbeb; margin: 10px 0;}
.fix  {padding: 12px 16px; border-left: 5px solid #dc2626; background: #fef2f2; margin: 10px 0;}
.exam {padding: 12px 16px; border-left: 5px solid #059669; background: #ecfdf5; margin: 10px 0;}
table {font-size: 95%;}
</style>

# 06 — Chi-Square Tests

### Inference for categorical counts

**Level:** beginner → advanced  
**Style:** short explanations, worked examples, formulas, runnable code, revision material

## What you will be able to do

- distinguish goodness-of-fit, independence, and homogeneity tests
- calculate expected counts and the chi-square statistic
- check degrees of freedom and expected-count assumptions
- use residuals and Cramér’s V to move beyond a p-value

## Resource coverage

- Transcript lines 3177–3836: categorical distributions, bike and handedness examples, 2010/2020 weight-category calculation
- Topic-map image lines 70–71 identifies the two lecture units: What is Chi-Square and Chi-Square Goodness of Fit

<div class="note"><b>How to study this notebook:</b> Read once without memorising. Then rerun the code, solve each checkpoint without looking, and finish with the cheat sheet.</div>


## 1. What chi-square tests work with

Chi-square tests use **counts in categories**.

Examples:

- left-handed versus right-handed;
- preferred bike colour;
- weight category;
- treatment group × recovered/not recovered.

They compare observed counts $O_i$ with counts expected under $H_0$, $E_i$:

$$\chi^2=\sum_i\frac{(O_i-E_i)^2}{E_i}$$

Properties:

- every contribution is non-negative;
- $\chi^2=0$ means observed equals expected exactly;
- large values mean stronger disagreement;
- the chi-square distribution is right-skewed and non-negative;
- the test is usually right-tailed.


## 2. Three closely related tests

| Test | One-sentence question | Expected count |
|---|---|---|
| Goodness of fit | Does one categorical variable follow specified proportions? | $E_i=np_i$ |
| Independence | Are two categorical variables associated in one population? | $E_{ij}=\frac{(row_i\ total)(col_j\ total)}{N}$ |
| Homogeneity | Do several populations share the same categorical distribution? | same contingency-table calculation |

The arithmetic for independence and homogeneity is the same; the sampling design and interpretation differ.


## 3. Goodness-of-fit hypotheses and degrees of freedom

If the null proportions are $p_1,\ldots,p_k$:

$$H_0:\text{population category proportions are }p_1,\ldots,p_k$$

$$H_a:\text{at least one category proportion differs}$$

Expected count:

$$E_i=np_i$$

If no parameters were estimated from the same data:

$$df=k-1$$

If $m$ parameters were estimated to create expected probabilities:

$$df=k-1-m$$


## 4. Lecture warm-ups

### Bike-colour theory

Theory: one-third prefer each of yellow, red, and orange. Observed sample counts are 22, 17, 59.

- Total $n=98$.
- Expected count in each category: $98/3\approx32.67$.
- The large orange count will contribute strongly to $\chi^2$.

### Handedness theory

Class of 75: observed 11 left-handed and 64 right-handed. Theory says 12% are left-handed.

Expected:

- Left: $0.12(75)=9$.
- Right: $0.88(75)=66$.

The statistic compares $(11,64)$ with $(9,66)$.


In [1]:
# Beginner-friendly guide:
# We load tools for calculations, probability distributions, tables, and pictures.
# The next examples compare category counts with the counts expected under a simple claim.
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)


In [2]:
# Beginner-friendly guide:
# This helper checks that totals match, then adds the squared observed-versus-expected differences.
# It also returns each category contribution so we can see which boxes matter most.
def chi_square_gof(observed, expected):
    observed = np.asarray(observed, dtype=float)
    expected = np.asarray(expected, dtype=float)
    if not np.isclose(observed.sum(), expected.sum()):
        raise ValueError('Observed and expected totals must match')
    contributions = (observed - expected)**2 / expected
    return contributions.sum(), contributions

chi_hand, parts_hand = chi_square_gof([11, 64], [9, 66])
p_hand = stats.chi2.sf(chi_hand, df=1)
print(f"Handedness: chi-square={chi_hand:.4f}, df=1, p={p_hand:.4f}")
print('Contributions:', parts_hand)


Handedness: chi-square=0.5051, df=1, p=0.4773
Contributions: [0.4444 0.0606]


## 5. Full lecture example: weight categories over time

2010 population proportions:

| Weight category | Null proportion |
|---|---:|
| below 50 kg | 0.20 |
| 50–75 kg | 0.30 |
| above 75 kg | 0.50 |

2020 sample of $n=500$ has observed counts $(140,160,200)$.

Expected counts from 2010:

$$E=(0.2\cdot500,0.3\cdot500,0.5\cdot500)=(100,150,250)$$

Hypotheses:

- $H_0$: the 2020 population has the 2010 proportions.
- $H_a$: at least one 2020 proportion differs.

Statistic:

$$\chi^2=\frac{(140-100)^2}{100}+\frac{(160-150)^2}{150}+\frac{(200-250)^2}{250}$$

$$=16+0.6667+10=26.6667$$

Degrees of freedom: $df=3-1=2$.

At $\alpha=0.05$, the critical value is 5.991. Because $26.67>5.991$, reject $H_0$. The p-value is about $1.62\times10^{-6}$.


In [3]:
# Beginner-friendly guide:
# We compare three observed weight-category counts with an expected pattern.
# The table shows each box contribution and a residual that tells whether it is above or below expectation.
observed = np.array([140, 160, 200])
expected = np.array([100, 150, 250])

chi_manual, contributions = chi_square_gof(observed, expected)
chi_scipy, p_scipy = stats.chisquare(observed, f_exp=expected)
residuals = (observed - expected) / np.sqrt(expected)

table = pd.DataFrame({
    'Observed': observed,
    'Expected': expected,
    'Chi contribution': contributions,
    'Pearson residual': residuals,
}, index=['<50', '50–75', '>75'])

print(table.round(4).to_string())
print(f"\nchi-square={chi_scipy:.4f}, df=2, p={p_scipy:.3e}")


       Observed  Expected  Chi contribution  Pearson residual
<50         140       100           16.0000            4.0000
50–75       160       150            0.6667            0.8165
>75         200       250           10.0000           -3.1623

chi-square=26.6667, df=2, p=1.620e-06


### Which categories drive the result?

The omnibus p-value says the distributions differ, but not where.

- Chi-square contributions show how much each cell adds.
- Pearson residuals $r_i=(O_i-E_i)/\sqrt{E_i}$ show direction and standardised size.
- Here, below-50 is much higher than expected and above-75 is much lower.

Inspecting residuals is the categorical equivalent of looking under the bonnet instead of merely admiring the check-engine light.


## 6. Independence test

For an $r\times c$ table:

$$H_0:\text{row and column variables are independent}$$

$$H_a:\text{they are associated}$$

Expected cell count:

$$E_{ij}=\frac{(\text{row }i\text{ total})(\text{column }j\text{ total})}{N}$$

Degrees of freedom:

$$df=(r-1)(c-1)$$

The test detects association, not causation. It also does not by itself describe the strength or pattern of association.


In [4]:
# Beginner-friendly guide:
# This small two-way table asks whether study method and pass-or-fail are independent.
# SciPy calculates the expected counts and the chi-square result under the no-relationship story.
# Example: study method x pass/fail
table = np.array([[42, 18],
                  [28, 32]])
chi2, p, df, expected = stats.chi2_contingency(table, correction=False)
print(f"chi-square={chi2:.4f}, df={df}, p={p:.4f}")
print('Expected counts under independence:')
print(np.round(expected, 2))


chi-square=6.7200, df=1, p=0.0095
Expected counts under independence:
[[35. 25.]
 [35. 25.]]


## 7. Assumptions and small counts

- Observations are independent.
- Categories are mutually exclusive and exhaustive.
- Use counts, not raw percentages, as input.
- Expected counts should be large enough for the chi-square approximation.

Common classroom rule:

- all expected counts at least 5.

More nuanced guidance often allows some below 5 if none is below 1 and no more than 20% are below 5. Rules vary by setting; inspect the table.

For a sparse $2\times2$ table, Fisher’s exact test is often appropriate. For larger sparse tables, exact or simulation-based methods may be needed.

Yates’ continuity correction is sometimes used for $2\times2$ tables; it is conservative and should be reported if applied.


## 8. Effect size: Cramér’s V

For an $r\times c$ contingency table:

$$V=\sqrt{\frac{\chi^2}{N\min(r-1,c-1)}}$$

- $V=0$: no association.
- Larger values: stronger association.
- Maximum is 1 for many tables, but interpretation depends on dimensions and context.

For goodness-of-fit, Cohen’s $w$ is common:

$$w=\sqrt{\sum_i\frac{(p_i-p_{0i})^2}{p_{0i}}}$$

The p-value answers “is there evidence of departure?” Effect size answers “how large is the departure?”


# End-of-topic cheat sheet

| Need | Formula / rule |
|---|---|
| Statistic | $\chi^2=\sum(O-E)^2/E$ |
| GOF expected | $E_i=np_i$ |
| GOF df | $k-1$ minus fitted parameters |
| Independence expected | row total × column total / N |
| Independence df | $(r-1)(c-1)$ |
| Decision | large $\chi^2$, small p → reject |
| Diagnostics | cell contributions and Pearson residuals |
| Association size | Cramér’s $V$ |
| Sparse 2×2 | consider Fisher’s exact test |

**Input is counts.** If you only have percentages, you also need the underlying total.


# Revision questions and answers

**Q1. What data type does a chi-square test use?**

<details><summary>Answer</summary>

Counts of observations in mutually exclusive categories.

</details>

---

**Q2. What does goodness of fit test?**

<details><summary>Answer</summary>

Whether one categorical variable follows a specified population distribution.

</details>

---

**Q3. How are GOF expected counts calculated?**

<details><summary>Answer</summary>

$E_i=np_i$.

</details>

---

**Q4. What is df for a three-category GOF test with no fitted parameters?**

<details><summary>Answer</summary>

$3-1=2$.

</details>

---

**Q5. Why is the chi-square rejection region usually on the right?**

<details><summary>Answer</summary>

The statistic is non-negative, and larger values mean greater observed–expected disagreement.

</details>

---

**Q6. What is the lecture weight statistic?**

<details><summary>Answer</summary>

$\chi^2=26.6667$ with df = 2.

</details>

---

**Q7. Does a significant independence test prove causation?**

<details><summary>Answer</summary>

No. It shows evidence of association under the design.

</details>

---

**Q8. What should you inspect after a significant test?**

<details><summary>Answer</summary>

Cell contributions/residuals and an effect size such as Cramér’s V.

</details>


## Friendly theory: chi-square asks whether count boxes look more different than chance expects

> **This is an extra, simple-language companion. The detailed notes above stay exactly as they are.**

Imagine sorting candies into colour boxes. A chi-square test compares the **observed** counts in the boxes with the **expected** counts we would see if a simple claim were true.

### The score idea

For each box, look at `observed - expected`. Big differences matter more, but we divide by the expected count so a difference is judged fairly. Then we add all the box contributions to make the chi-square score.

### Two common questions

- **Goodness of fit:** Do these counts match one expected pattern?
- **Independence:** Are two category labels related, such as study method and pass/fail?

### Remember it

**Chi-square uses counts, not raw measurements.** Very tiny expected counts can make the usual chi-square rule unreliable.
